# LLM4Series — Forecasting with an LLM

This notebook shows the end-to-end workflow for using a large language model
to forecast a time series with `llm4series`: data preparation, prompt construction,
model inference, result inspection, and evaluation.

## Step 1 — Import the library and configure the LLM

`ls.LLM` wraps any compatible language model. Pass the model name and
your API key. The key is read from an environment variable to keep credentials
out of the notebook.

In [ ]:
import os
import llm4series as ls

model = ls.LLM(model="gpt-5", api_key=os.getenv("OPENAI_API_KEY"))

## Step 2 — Load and preprocess the data, then split into train and test

Three preprocessing steps are applied conditionally:
- `impute_interpolate` fills missing values only if they exist.
- `agg_duplicates` collapses duplicate timestamps only if they exist.

`ts.split` creates a train window and a test window. `start` and `end`
define the training period; `periods=24` sets the forecast horizon (24 steps ahead).

In [ ]:
ts = ls.read_file("data/ETTh2.csv", index_col="date")
ts = ts.impute_interpolate("linear") if ts.isna().sum() > 0 else ts
ts = ts.agg_duplicates(method="sum") if ts.index.duplicated().sum() > 0 else ts

train, test = ts.split(
    start="2016-07-12 06:00:00",
    end="2016-09-20 05:00:00",
    periods=24
)

## Step 3 — Build the prompt

`ls.prompt` converts the training data into a structured prompt.
- `type='few_shot'` includes example input-output pairs to guide the model.
- `tsformat='csv'` serializes the time series as comma-separated text.
- `tstype='numeric'` signals that the values are continuous numbers.
- `examples=3` includes three demonstration windows.
- `forecast_horizon=24` tells the model how many steps to predict.

In [ ]:
prompt_config = ls.prompt(
    type="few_shot",
    ts=train,
    tsformat="csv",
    tstype="numeric",
    examples=3,
    forecast_horizon=24
)

## Step 4 — Run the forecast

`model.predict` sends the prompt to the LLM and returns a response object.
The library handles serialization, API calls, and parsing automatically.

In [ ]:
response = model.predict(prompt_config)

## Step 5 — Inspect the prediction

`response.prediction` returns the forecast as a time-indexed DataFrame,
ready for comparison with the test set.

In [ ]:
response.prediction

## Step 6 — Check latency and token usage

`response.time` gives total inference time in seconds.
`response.input_tokens` and `response.output_tokens` report token counts,
which are directly linked to API cost.

In [ ]:
print(response.time)
print(response.input_tokens)
print(response.output_tokens)

## Step 7 — Plot forecast vs. actual

`ls.plot` accepts two series and labels them via `groups`.
This lets you visually compare the real test values against the LLM forecast.

In [ ]:
ls.plot(
    test,
    response.prediction,
    kind='line',
    groups=["Real", "Predicted"],
    title="Forecast vs Actual",
    xlabel="Time",
    ylabel="Value"
)

## Step 8 — Evaluate forecast accuracy

`ls.metrics` computes SMAPE, MAE, and RMSE between the predicted and actual series.
Here it is applied to a single variable (`OT`) for a focused evaluation.

In [ ]:
ls.metrics(response.prediction["OT"], test["OT"])